In [6]:
import pandas as pd
import numpy as np
import os
import time
from tqdm.auto import tqdm
import duckdb
import sys

sys.path.append(os.path.dirname(os.getcwd()))
import importlib
import utils.data_preprocessing as du

importlib.reload(du)

<module 'utils.data_preprocessing' from '/home/jovyan/_shared_storage/temp_stud/MA_Otto/utils/data_preprocessing.py'>

In [2]:
# Initialize notebook

# get parent dict
parent = os.path.dirname(os.getcwd())

# set up duckdb connection
con = duckdb.connect()
con.execute("SET enable_progress_bar = false")

In [3]:
# load fut-data into memory, as it is used across all symbols
fut_dict = {}

for date in tqdm(du.SAMPLE_DATES, desc="Dates"):
    fut_df = con.execute(f"""
        SELECT "Timestamp", "Timestamp_Europe/Berlin", "MicroPrice_tick-based_10_1s" AS MicroPrice
        FROM read_csv('{parent}/data/raw/FUT_DAX Futures/FUT_DAX Futures_{date}.csv.gz')
    """).df()

    # filter needs the tz-aware datetime (Berlin wall-clock); resample_last then
    # collapses to one row per 100ms on the int-ns axis, cutting the feature-matrix
    # cost. Timestamps are not floored, so the later backward merge_asof is leak-free.
    fut_df = du.filter_trading_hours(fut_df)
    fut_df = du.resample_last(fut_df, "Timestamp", "100ms")

    fut = du.compute_feature_target_matrix(
        df=fut_df,
        ts_col='Timestamp',
        target_cols=[],
        feature_cols=['MicroPrice'],
        horizons=['-5m', '-2.5m', '-1m', '-30s', '-15s', '-5s', '-2s', '-1s', '-100ms'])

    fut_dict[date] = fut

#pd.concat([df.assign(date=key) for key, df in fut_dict.items()]).to_parquet(f"{parent}/data/processed/fut_dict.parquet", index=False)

Dates:   0%|          | 0/127 [00:00<?, ?it/s]

In [7]:
cols = ["Timestamp", "Timestamp_Europe/Berlin", "L1-BidPrice", "L1-AskPrice", "Side",
        "L1-QImb", "MidPrice", "MidPriceQW", "MidPriceCQW"]

symbols = [s for s in du.SYMBOLS if s != 'FUT_DAX Futures']
for symbol in tqdm(symbols, desc="Symbols"):

    os.makedirs(f'{parent}/data/processed/{symbol}', exist_ok=True)

    for date in tqdm(du.SAMPLE_DATES, desc="Dates", leave=False):
        path = f'{parent}/data/raw/CS_{symbol}/CS_{symbol}_{date}.csv.gz'

        group = con.execute(f"""
            SELECT {", ".join(f'"{c}"' for c in cols)}, "MicroPrice_tick-based_10_1s" AS MicroPrice
            FROM read_csv('{path}')
        """).df()

        filtered = du.filter_trading_hours(group).copy()

        filtered['TransactionPrice'] = du.compute_transaction_price(filtered)

        filtered = du.resample_last(filtered, "Timestamp", "100ms")

        featured = du.compute_feature_target_matrix(
            df=filtered,
            ts_col='Timestamp',
            target_cols=du.PRICE_MEASURES,
            feature_cols=['L1-QImb', 'MicroPrice'],
            horizons=['-5m', '-2.5m', '-1m', '-30s', '-15s', '-5s', '-2s', '-1s', '-100ms',
                        '100ms', '1s', '2s', '5s', '15s', '30s', '1m', '2.5m', '5m'])

        # merge_asof requires both sides sorted on the key (Timestamp, int ns).
        # Both come out of resample_last / compute_feature_target_matrix sorted,
        # so this is only a safety net.
        featured = featured.sort_values("Timestamp").reset_index(drop=True)

        merged = pd.merge_asof(
            featured,
            fut_dict[date].drop(columns="Timestamp_Europe/Berlin"),
            on="Timestamp",
            direction="backward",
            suffixes=("_LOB", "_FUT"))

        merged = merged.drop(columns=['TransactionPrice', 'MidPrice', 'MidPriceQW', 'MidPriceCQW', 'MicroPrice_FUT', 'MicroPrice_LOB',
                                        'Timestamp_Europe/Berlin', 'L1-BidPrice', 'L1-AskPrice', 'Side'])
        merged.to_parquet(f'{parent}/data/processed/{symbol}/{date}.parquet')

Symbols:   0%|          | 0/21 [00:00<?, ?it/s]

Dates:   0%|          | 0/127 [00:00<?, ?it/s]

Dates:   0%|          | 0/127 [00:00<?, ?it/s]

Dates:   0%|          | 0/127 [00:00<?, ?it/s]

Dates:   0%|          | 0/127 [00:00<?, ?it/s]

...

Dates:   0%|          | 0/127 [00:00<?, ?it/s]

Dates:   0%|          | 0/127 [00:00<?, ?it/s]

Dates:   0%|          | 0/127 [00:00<?, ?it/s]

Dates:   0%|          | 0/127 [00:00<?, ?it/s]

IOException: IO Error: No files found that match the pattern "/home/jovyan/_shared_storage/temp_stud/MA_Otto/data/raw/CS_FUT_DAX Futures/CS_FUT_DAX Futures_2023-01-02.csv.gz"

LINE 3:             FROM read_csv('/home/jovyan/_shared_storage/temp_stud/MA_Otto...
                         ^

In [ ]:
test = pd.read_parquet(f"{parent}/data/processed/Adidas/2023-01-02.parquet")
test

In [ ]:
test.columns